In [ ]:
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt

# ----------- 1. 데이터 로딩 ----------- #
# df = pd.read_csv(r"C:\Users\seung\OneDrive\주식\Back Test\Tesla\Tesla Stock Price History_지표포함.csv", encoding="utf-8-sig")
df = pd.read_csv(r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv", encoding="utf-8-sig")

df["날짜"] = pd.to_datetime(df["날짜"])

#----------- 2. 조건 클래스 정의 ----------- #
class StrategyCondition:
    def check(self, row): raise NotImplementedError()

class RSIBelow(StrategyCondition):
    def __init__(self, threshold): self.threshold = threshold
    def check(self, row): return row.get("RSI (14일)", 100) < self.threshold

class BollingerNearLower(StrategyCondition):
    def __init__(self, buffer_pct): self.buffer_pct = buffer_pct
    def check(self, row): return row['종가'] < row.get("볼린저밴드 하단", row['종가']) * (1 + self.buffer_pct)

class MACDPositive(StrategyCondition):
    def check(self, row): return row.get("MACD", 0) > row.get("MACD 시그널", 0)

class MA5AboveMA10(StrategyCondition):
    def check(self, row): return row.get("SMA 5일", 0) > row.get("SMA 10일", 0)

class MA5BelowMA60(StrategyCondition):
    def check(self, row): return row.get("SMA 5일", 0) < row.get("SMA 60일", 0)

class TwoWeekPriceLow(StrategyCondition):
    def __init__(self, max_pct): self.max_pct = max_pct
    def check(self, row): return row.get("가격 상승률 (2주)", 100) < self.max_pct

class RSISell(StrategyCondition):
    def __init__(self, threshold): self.threshold = threshold
    def check(self, row): return row.get("RSI (14일)", 0) > self.threshold

# ----------- 3. 전략 클래스 ----------- #
class CompositeStrategy:
    def __init__(self, buy_conditions, sell_conditions):
        self.buy_conditions = buy_conditions
        self.sell_conditions = sell_conditions
        self.buy_dates = []
        self.sell_dates = []

    def should_buy(self, row):
        if all(cond.check(row) for cond in self.buy_conditions):
            self.buy_dates.append(pd.to_datetime(row['날짜']))
            return True
        return False

    def should_sell(self, row):
        if any(cond.check(row) for cond in self.sell_conditions):
            self.sell_dates.append(pd.to_datetime(row['날짜']))
            return True
        return False

# ----------- 4. 백테스트 클래스 ----------- #
class BacktestPeriodModified:
    def __init__(self, df, start_date, end_date, strategy=None, allow_sell=True):
        self.df = df[(df['날짜'] >= start_date) & (df['날짜'] < end_date)].copy()
        self.cash = 10_000.0
        self.shares = 0.0
        self.total_cost = 0.0
        self.cooldown = 0
        self.strategy = strategy
        self.start = pd.to_datetime(start_date)
        self.end = pd.to_datetime(end_date)
        self.allow_sell = allow_sell
        self.buy_lots = []
        self.buy_prices = []
        self.sell_prices = []

    def run(self):
        if self.strategy:
            self.strategy.buy_dates = []
            self.strategy.sell_dates = []

        for _, row in self.df.iterrows():
            price = row['종가']
            if self.cooldown > 0:
                self.cooldown -= 1
                continue

            if self.strategy is None:
                if self.shares == 0:
                    self._buy(price, row['날짜'])
            else:
                if self.strategy.should_buy(row):
                    self._buy(price, row['날짜'])
                elif self.allow_sell and self.strategy.should_sell(row) and self.shares > 0:
                    self._sell(price, row['날짜'])

        return self._summary()

    def _buy(self, price, date):
        if self.cash > 0:
            buy_amount = self.cash * 0.1
            num_shares = buy_amount / price
            self.shares += num_shares
            self.cash -= buy_amount
            self.total_cost += buy_amount
            self.buy_lots.append((date, price, num_shares))
            self.buy_prices.append((date, round(price, 2)))
            self.cooldown = 5

    def _sell(self, price, date):
        if self.shares > 0:
            proceeds = self.shares * price
            self.cash += proceeds
            self.shares = 0
            self.total_cost = 0
            self.buy_lots.clear()
            self.sell_prices.append((date, round(price, 2)))
            self.cooldown = 5

    def _summary(self):
        final_value = self.cash + self.shares * self.df.iloc[-1]['종가']
        roi = (final_value - 10_000) / 10_000 * 100
        value_to_cost = final_value / self.total_cost if self.total_cost > 0 else np.nan
        return {
            "시작일": self.start.date(),
            "종료일": self.end.date(),
            "최종 자산": round(final_value, 2),
            "투자 비용": round(self.total_cost, 2),
            "수익률 (%)": round(roi, 2),
            "자산/매입비용": round(value_to_cost, 2) if not np.isnan(value_to_cost) else "N/A",
            "매수일 수": len(self.strategy.buy_dates) if self.strategy else "N/A",
            "매도일 수": len(self.strategy.sell_dates) if self.strategy else "N/A",
            "매수 날짜": [d.strftime('%Y-%m-%d') for d in self.strategy.buy_dates] if self.strategy else [],
            "매수 가격": [f"{d.strftime('%Y-%m-%d')}: ${p}" for d, p in self.buy_prices],
            "매도 날짜": [d.strftime('%Y-%m-%d') for d in self.strategy.sell_dates] if self.strategy else [],
            "매도 가격": [f"{d.strftime('%Y-%m-%d')}: ${p}" for d, p in self.sell_prices],
            "전략": "조건 전략" if self.strategy else "베이스라인"
        }

# ----------- 5. 최적화 실행 ----------- #
periods = [("2022-01-01", "2023-01-01"), ("2023-01-01", "2024-01-01"), ("2024-01-01", "2025-01-01")]
rsi_range = range(30, 60, 5)
bollinger_buffer_range = [0.01, 0.015, 0.02, 0.025]
two_week_range = [1, 2, 3, 4, 5]

optimization_results = []
macd = MACDPositive()
ma5_above_ma10 = MA5AboveMA10()
ma5_below_ma60 = MA5BelowMA60()
rsi_sell = RSISell(70)

for rsi_thres, bb_buf, two_week_pct in itertools.product(rsi_range, bollinger_buffer_range, two_week_range):
    buy_conditions = [
        RSIBelow(rsi_thres),
        BollingerNearLower(bb_buf),
        macd,
        ma5_above_ma10,
        ma5_below_ma60,
        TwoWeekPriceLow(two_week_pct)
    ]
    strategy_with_sell = CompositeStrategy(buy_conditions, [rsi_sell])
    strategy_no_sell = CompositeStrategy(buy_conditions, [])

    total_roi_with_sell = 0
    total_roi_no_sell = 0
    full_results_with_sell = []
    full_results_no_sell = []

    for start, end in periods:
        r1 = BacktestPeriodModified(df, start, end, None, allow_sell=False).run()  # baseline1
        r2 = BacktestPeriodModified(df, start, end, strategy_no_sell, allow_sell=False).run()  # baseline2
        r3 = BacktestPeriodModified(df, start, end, strategy_with_sell, allow_sell=True).run()  # full strategy

        total_roi_with_sell += r3["수익률 (%)"]
        full_results_with_sell.append(r3)

        total_roi_no_sell += r2["수익률 (%)"]
        full_results_no_sell.append(r2)

    optimization_results.append({
        "RSI 조건": rsi_thres,
        "볼린저 하단 버퍼": bb_buf,
        "2주 상승률 조건": two_week_pct,
        "평균 수익률 (%)": round(total_roi_with_sell / len(periods), 2),
        "전략 결과 (매수+매도)": full_results_with_sell,
        "전략 결과 (매수만)": full_results_no_sell
    })

# ----------- 6. 시각화 ----------- #
optimization_df = pd.DataFrame(optimization_results).sort_values("평균 수익률 (%)", ascending=False).head(5)

plt.figure(figsize=(12, 6))
plt.barh(optimization_df.index.astype(str), optimization_df["평균 수익률 (%)"])
plt.xlabel("평균 수익률 (%)")
plt.ylabel("전략 인덱스")
plt.title("Top 5 전략 수익률 비교")
plt.tight_layout()
plt.gca().invert_yaxis()
plt.show()

# ----------- 7. 저장 ----------- #
detailed_rows = []
for _, row in optimization_df.iterrows():
    for res in row["전략 결과 (매수+매도)"]:
        combined = res.copy()
        combined.update({
            "RSI 조건": row["RSI 조건"],
            "볼린저 하단 버퍼": row["볼린저 하단 버퍼"],
            "2주 상승률 조건": row["2주 상승률 조건"],
            "전략 유형": "조건 전략"
        })
        detailed_rows.append(combined)

for _, row in optimization_df.iterrows():
    for res in row["전략 결과 (매수만)"]:
        combined = res.copy()
        combined.update({
            "RSI 조건": row["RSI 조건"],
            "볼린저 하단 버퍼": row["볼린저 하단 버퍼"],
            "2주 상승률 조건": row["2주 상승률 조건"],
            "전략 유형": "조건 전략 (매도 없음)"
        })
        detailed_rows.append(combined)

pd.DataFrame(detailed_rows).to_csv(
    r"C:\Users\LabPC\OneDrive\주식\Results\Top_전략_상세결과.csv",
    index=False, encoding="utf-8-sig"
)
